# Part 8 · Notebook 06 — Walk-forward analysis and purged cross-validation

**Sessions:** S11 (Walk-forward analysis) · S12 (Purged k-fold & CPCV) · [Lesson plan](../../docs/lessons/PART_08_BACKTESTING_RISK_PORTFOLIO.md) · graded labs in [`labs/part08/`](../../labs/part08/)

**You will:**
1. Generate rolling walk-forward windows, and measure walk-forward efficiency.
2. See a model that knows only the date score R² = 0.89 on pure noise with shuffled k-fold.
3. Purge and embargo the folds, and the leak disappears.
4. Count the backtest paths combinatorial purged CV produces.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known truth (known regimes, known Sharpe ratios, pure noise), so every statistic can be checked against reality and every discovery against luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p8lib.py is in notebooks/part08/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p8lib as p

p.use_course_style()

## 1. Walk-forward windows

Optimize on `train` bars, trade the next `test` bars, move forward by `test`, repeat. Rolling windows keep `train` bars; anchored ones start at 0. Yield `(train_idx, test_idx)` as integer arrays, and stop when a full test window no longer fits.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def walk_forward(n, train, test, anchored=False):
    start = 0
    while start + train + test <= n:
        tr0 = 0 if anchored else start
        yield ...                                 # ✍️ (train indices, test indices) as np.arange
        start += test

cases = [(20, 8, 4, False), (20, 8, 4, True), (3000, 750, 250, False)]
mine = [p.attempt(lambda *a: [(tr.tolist(), te.tolist()) for tr, te in walk_forward(*a)], *cs) for cs in cases]
mine = p.check("walk_forward", mine, [[(tr.tolist(), te.tolist()) for tr, te in p.walk_forward(*cs)] for cs in cases])
for tr, te in mine[0]:
    print(f"train {tr[0]:2d}–{tr[-1]:2d}   test {te[0]:2d}–{te[-1]:2d}")

In each window, pick the TSMOM parameters with the best in-sample Sharpe, then record what they earn out of sample. **Walk-forward efficiency** (WFE) = mean OOS Sharpe / mean IS Sharpe: near 1 means the in-sample choice carried over; near 0 means it didn't.

In [ ]:
bars = p.regime_market()
o, c = bars.open.to_numpy(), bars.close.to_numpy()
wf = p.walk_forward_optimize(lambda lookback: p.first_look_pnl(p.tsmom_signal(c, lookback), o),
                             {"lookback": [40, 80, 120, 180, 250]}, len(c), 750, 250)
display(pd.DataFrame({"chosen lookback": [d["lookback"] for d in wf["params"]], "in-sample Sharpe": wf["is_sharpe"],
                      "out-of-sample Sharpe": wf["oos_sharpe"]}).round(2).T)
print(f"stitched out-of-sample Sharpe {p.sharpe(wf['oos']):.2f}; walk-forward efficiency {wf['wfe']:.2f}")

## 2. Why ordinary k-fold lies on financial data

A typical label is a **forward return** over several days, so neighbouring labels overlap. Here the returns are pure noise (nothing is predictable) and each label is the sum of the next 20 daily returns. The "model" is a 1-nearest-neighbour whose only feature is the **date**: it predicts each test label with the label of the closest training day. It can only succeed by leakage.

In [ ]:
r, y = p.overlapping_labels(horizon=20)
for name, splits in [("shuffled k-fold (sklearn KFold(shuffle=True))", p.shuffled_kfold(len(y))),
                     ("contiguous k-fold, no purging", p.contiguous_kfold(len(y)))]:
    print(f"{name:46s} out-of-sample R² = {p.nearest_neighbour_r2(y, splits):+.2f}")

Shuffled k-fold gives a model with **no information** an R² of about 0.9: every test day has a training neighbour whose label shares 19 of its 20 days. Contiguous folds leak only at their edges.

## 3. Purging and embargo

With contiguous test folds, **purge** every training sample whose label window `[i, i + label_horizon]` reaches into the fold (keep `i` only if `i + label_horizon < fold start`), and **embargo** the `ceil(embargo × n)` samples right after the fold (keep `i` only if `i > fold end + emb`). Yield `(train_idx, test_idx)`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def purged_kfold(n, k=5, label_horizon=5, embargo=0.01):
    emb = int(np.ceil(embargo * n))
    idx = np.arange(n)
    for test in np.array_split(idx, k):
        lo, hi = test[0], test[-1]
        keep = ...                                # ✍️ boolean mask of the training samples to keep
        yield idx[keep], test

mine = p.attempt(lambda: [(tr.tolist(), te.tolist()) for tr, te in purged_kfold(100, 4, 5, 0.03)])
mine = p.check("purged_kfold", mine, [(tr.tolist(), te.tolist()) for tr, te in p.purged_kfold(100, 4, 5, 0.03)])
for tr, te in (mine[1:3]):
    gap = [i for i in range(100) if i not in tr and i not in te]
    print(f"test {te[0]}–{te[-1]}: removed from training {gap}")
print(f"purged 5-fold, horizon 20:  out-of-sample R² = {p.nearest_neighbour_r2(y, p.purged_kfold(len(y), 5, 20, 0.01)):+.2f}")

The leak is gone: the date-only model is now (correctly) worse than predicting the mean.

## 4. Combinatorial purged CV

Walk-forward gives **one** out-of-sample path, so one lucky or unlucky period decides everything. CPCV splits the data into `N` groups and tests on every combination of `k` of them (purged as above). Each group is tested `C(N−1, k−1)` times, so the out-of-sample results assemble into `C(N, k)·k / N` complete backtest paths.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from math import comb

def cpcv_n_paths(n_groups, k_test):
    return ...                                    # ✍️ an int

cases = [(6, 2), (10, 2), (10, 3), (12, 4)]
mine = [p.attempt(cpcv_n_paths, *cs) for cs in cases]
mine = p.check("cpcv_n_paths", mine, [p.cpcv_n_paths(*cs) for cs in cases])
pd.DataFrame({"groups N": [a for a, _ in cases], "test groups k": [b for _, b in cases], "splits C(N,k)": [comb(a, b) for a, b in cases],
              "backtest paths": mine})

In [ ]:
splits = p.cpcv_splits(600, n_groups=6, k_test=2, label_horizon=10, embargo=0.02)
grid_img = np.zeros((len(splits), 600))
for row, (train, combo, test) in enumerate(splits):
    grid_img[row, train] = 1; grid_img[row, test] = 2
plt.figure(figsize=(10, 4)); plt.imshow(grid_img, aspect="auto", cmap="Greys", interpolation="nearest")
plt.xlabel("sample"); plt.ylabel("split"); plt.title("CPCV (N=6, k=2): black = test, grey = train, white = purged or embargoed"); plt.show()

## Wrap-up

* Walk-forward mimics how the strategy would really have been re-fitted; report OOS results and the WFE.
* Never shuffle time series; purge overlapping labels and embargo after each test fold.
* CPCV turns one OOS path into many, so you see a distribution instead of one draw.
* Graded version: `labs/part08/week27_optimization` (walk-forward optimizer, purged k-fold, CPCV splits).